# Part 1: Data Retrieval

### **Purpose:** This notebook is meant to retrieve and process data while in the AE projection(s) for the current data request. It's intention is to create intermediate data products for the various desired metrics to be reformatted and exported properly in `data_reprojection_and_export.ipynb`.

In [8]:
%reload_ext autoreload
%autoreload 2

In [9]:
pip install pydrive2

Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys
import os
import geopandas as gpd
from climakitae.core.data_load import load
from climakitae.core.data_export import export
from climakitae.util.utils import add_dummy_time_to_wl
from pathlib import Path
import xarray as xr
import numpy as np
import warnings
from typing import Tuple, Optional
from matplotlib.colors import Colormap, ListedColormap
from matplotlib.gridspec import GridSpec
import cartopy.io.img_tiles as cimgt
import cartopy.crs as ccrs


warnings.filterwarnings("ignore", category=RuntimeWarning)

# Add the parent directory to sys.path
sibling_path = (Path.cwd().parent.parent / "src").resolve()
if str(sibling_path) not in sys.path:
    sys.path.append(str(sibling_path))

# Now you can import
from data_access import find_lat_lon_bounds, retrieve_clipped_data
from data_cleaning import add_wrf_crs, get_wrf_crs
from metric_calc import calc_thresh_pairs
from plotting import (
    plot_metric_deltas,
    plot_metric_by_wl,
    calc_extent,
    image_spoof,
    lighter_r_rev,
    lighter_r,
)
from export import export_shapefile_zip, export_reproj_data, combine_exported_shapefiles

In [16]:
def count_streaks(arr, duration):
    """
    Return the number of 'duration' length streaks in the dataset.

    Parameters:
    -----------
    arr (np.ndarray): Array at single gridpoint
    duration (int): Length of streak

    Returns:
    --------
    xr.DataArray: Array containing counts of streaks
    """
    if len(arr) < duration:
        return 0

    arr = np.asarray(arr, dtype=bool)
    kernel = np.ones(duration, dtype=int)
    rolling = np.convolve(arr, kernel, mode="valid")
    full_hits = rolling == duration

    count = 0
    i = 0
    while i < len(full_hits):
        if full_hits[i]:
            count += 1
            i += duration
        else:
            i += 1
    return count


def calc_freq(
    data: xr.DataArray,
    threshold: float,
    duration: int,
    comparison: str = "greater",
    reduce_func: str = "median",
    groupby: str = "year",
    upper_threshold: Optional[float] = None,
) -> xr.DataArray:
    """Get frequency of data by threshold.

    Parameters:
    -----------
    data (xr.DataArray): Data
    threshold (float): Threshold value (lower threshold if comparison is between)
    duration (int): Length of streak, same as time units
    comparison (str): "greater", "less", or "between"
    reduce_func (str): "median" or "sum"
    groubpy (str): "year" or "season"
    upper_threshold (float): Required if comparison is "between"

    Returns:
    xr.DataArray: Counts of streaks within threshold
    """

    if groupby not in ["year", "season"]:
        raise ValueError("groupby must be 'year' or 'season'")

    # Apply comparison
    if comparison == "greater":
        mask = data > threshold
    elif comparison == "less":
        mask = data < threshold
    elif comparison == "between":
        if upper_threshold is None:
            raise ValueError(
                "upper_threshold must be provided when comparison is 'between'"
            )
        mask = (data > threshold) & (data < upper_threshold)
    else:
        raise ValueError("comparison must be 'greater', 'less', or 'between'")

    # Get non-NaN spatial mask
    spatial_mask = ~data.isnull().all(dim="time").isel(simulation=0)

    # Group by year or season
    group_dim = f"time.{groupby}"
    grouped = mask.groupby(group_dim)

    # Count streaks within each group
    streak_counts = grouped.apply(
        lambda group: xr.apply_ufunc(
            count_streaks,
            group,
            kwargs={"duration": duration},
            input_core_dims=[["time"]],
            output_core_dims=[[]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[int],
        )
    )

    # Average over simulations and time groups (if yearly, else leave the seasons in)
    if groupby == "year":
        dims_to_reduce = ["year", "simulation"]
        result = streak_counts
    elif groupby == "season":
        dims_to_reduce = ["simulation"]
        result = streak_counts
    else:
        raise ValueError("groupby must be 'year' or 'season'")

    # Reduce across the right dimensions
    if reduce_func == "median":
        metric = result.median(dim=dims_to_reduce)
    elif reduce_func == "sum":
        metric = result.sum(dim=dims_to_reduce)
    else:
        raise ValueError("reduce_func must be 'median' or 'sum'")

    return metric.where(spatial_mask)


def calc_thresh_pairs(
    da, threshold_pairs, duration, groupby="year", reduce_func="median"
):
    """Run calc_freq for a pair of thresholds.

    Parameters:
    -----------
    da (xr.DataArray): Data
    threshold_pairs (tuple): Upper and lower threshold.
    duration (int): Duration of streak to count
    groupby (str): "year" or "season"
    reduce_func (str): "median" or "sum"

    Returns:
    xr.DataArray: Contains data for both pairs
    """
    calc_datas = xr.concat(
        [
            calc_freq(
                data=da,
                threshold=low,
                duration=duration,
                comparison="between",
                groupby=groupby,
                reduce_func=reduce_func,
                upper_threshold=high,
            )
            for (low, high) in threshold_pairs
        ],
        dim=xr.DataArray(
            [f"{low}-{high}" for (low, high) in threshold_pairs],
            dims="threshold_pair",
            name="threshold_pair",
        ),
    )
    return calc_datas

In [17]:
def plot_threshold_maps(
    calc_data: xr.DataArray,
    duration: int,
    gdf: gpd.geodataframe.GeoDataFrame,
    longitude: tuple[float, float],
    latitude: tuple[float, float],
    lighter_r_rev: ListedColormap,
    hist_wl: float,
    season: Optional[str] = None,
    threshold_low: Optional[float] = None,
    threshold_high: Optional[float] = None,
    show_plots: bool = True,
    lat_radius=60000,
    lon_radius=60000,
    lat_shift=0,
    lon_shift=0,
):
    """Create maps of threshold data

    Parameters:
    -----------
    calc_data (xr.DataArray): Data to plot
    duration (int): Duration of streak
    gdf (geopandas.geodataframe.GeoDataFrame): shape of region
    longitude (tuple[float, float]): longitude bounds
    latitude (tuple[float, float]): latitude bounds
    lighter_r_rev (ListedColormap): colormap
    hist_wl (float): Baseline warming level
    season (str): season
    threshold_low (float): upper threshold for visualization
    threshold_high (float): lower threshold for visualization
    """
    season_str = "annual"
    if season is not None:
        season_str = f"for {season}"

    # Do some set up for file name strings
    match (duration, season):
        case (1, None):
            num = 1
        case (3, None):
            num = 2
        case (1, list):
            num = 3
        case (3, list):
            num = 4

    for pair in calc_data.threshold_pair.values:
        one_data = calc_data.sel(threshold_pair=pair)
        temp_range = str(pair)  # or custom formatting depending on type

        cmap_lim = get_cbar_lim(one_data)

        cmap_one_max = one_data.max().values
        if cmap_one_max == 0:
            cmap_one_max = 1

        # Plot main warming level map
        plot_metric_by_wl(
            one_data,
            f"Avg Number of {duration}-Day Consecutive Heating \n Events Per Year Between NOAA Heat Index {temp_range}°F {season_str}",
            gdf,
            longitude,
            latitude,
            f"Events per Year {season_str}",
            lighter_r_rev,
            save_name=f"figures/heat_index/{num}_{region_name}_extreme_heat_{temp_range}_{duration}day_count_{season_str}",
            nrows=2,
            ncols=2,
            colorbar_min=0,
            colorbar_max=cmap_one_max,
            basemap_style="map",
            threshold_low=threshold_low,
            threshold_high=threshold_high,
            lat_radius=lat_radius,
            lon_radius=lon_radius,
            lat_shift=lat_shift,
            edgecolor="black",
            # show_plot=show_plots
        )

        # Plot difference across warming levels
        plot_metric_deltas(
            one_data,
            f"Difference in Avg Number of {duration}-Day Consecutive Heating \n Events Per Year Between NOAA Heat Index {temp_range}°F between Future WL and WL {hist_wl} {season_str}",
            hist_wl,
            gdf,
            longitude,
            latitude,
            f"Difference per Year {season_str}",
            "PuOr_r",
            save_name=f"figures/heat_index/{num}_{region_name}_extreme_heat_{temp_range}_{duration}day_diff_{season_str}",
            nrows=1,
            ncols=3,
            colorbar_min=-cmap_lim,
            colorbar_max=cmap_lim,
            basemap_style="map",
            threshold_low=threshold_low,
            threshold_high=threshold_high,
            lat_radius=lat_radius,
            lon_radius=lon_radius,
            lat_shift=lat_shift,
            edgecolor="black",
            # show_plot=show_plots
        )

In [18]:
def get_cbar_lim(mydata):
    """Return colorbar limits for difference plots.

    Parameters
    ----------
    mydata (xr.DataArray or xr.Dataset): data being plotted.

    Returns
    -------
    cbar_lim (float): colorbar limit value
    """
    # Depending on metric and sign of change, need to check both
    # the closest and furthest warming level to historical
    min_diff1 = (
        (
            mydata.sel(warming_level=mydata.warming_level.values[1])
            - mydata.sel(warming_level=mydata.warming_level.values.min())
        )
        .min()
        .load()
    )
    min_diff2 = (
        (
            mydata.sel(warming_level=mydata.warming_level.values.max())
            - mydata.sel(warming_level=mydata.warming_level.values.min())
        )
        .min()
        .load()
    )
    max_diff1 = (
        (
            mydata.sel(warming_level=mydata.warming_level.values.max())
            - mydata.sel(warming_level=mydata.warming_level.values.min())
        )
        .max()
        .load()
    )
    max_diff2 = (
        (
            mydata.sel(warming_level=mydata.warming_level.values[1])
            - mydata.sel(warming_level=mydata.warming_level.values.min())
        )
        .max()
        .load()
    )
    cbar_lim = max(
        abs(max_diff1.data),
        abs(max_diff1.data),
        abs(min_diff1.data),
        abs(min_diff2.data),
    )
    return cbar_lim

### Setup output directories

In [19]:
os.makedirs("Exports/heat_index", exist_ok=True)
os.makedirs("figures/heat_index", exist_ok=True)

### Setting global variables

In [24]:
# Saving shapefile of desired area
shapefile_path = (
    "../geometries/PajaroRiverWatershed/PajaroRiverWatershed.shp"
)
#shapefile_path_figure = "/IRWD.shp"

# Read in shapefile
gdf = gpd.read_file(shapefile_path)
#gdf_viz = gpd.read_file(shapefile_path_figure)

In [25]:
# Region name for figures
region_name = "IRWD"

# Saving CRS as a string
# dest_crs = "ESRI:102645"
dest_crs = get_wrf_crs()

# Resolution of data (needed to add buffer for data retrieval)
resolution = "3 km"

# Find latitude and longitude of region of interest
latitude, longitude = find_lat_lon_bounds(resolution, gdf)

# Downscaling method
downscaling_method = "Dynamical"

# Warming levels
warming_levels = [0.8, 1.5, 2.0, 3.0]
hist_wl = warming_levels[0]

# Dictionary between season month abbreviations and season names
seasons_dict = dict(
    zip(["DJF", "JJA", "MAM", "SON"], ["winter", "summer", "spring", "fall"])
)

# other plot settings
# Having a roughly 3:5 aspect ratio (lat:lon radius) works best
lat_radius = 50000
lon_radius = 100000
lat_shift = -0.04
edgecolor = "black"

# Metrics

## 1. Extreme Heat Events Frequency

## Save for all warming levels

In [26]:
"""for wl in warming_levels[-1:]:
## Retrieving data and sanity check plotting
da1 = retrieve_clipped_data(
    "NOAA Heat Index",
    downscaling_method,
    resolution,
    "hourly",
    "degF",
    [wl],
    gdf,
    show_plots=True,
    latitude=latitude,
    longitude=longitude,
)
loaded_daily_data = load(
    add_dummy_time_to_wl(da1).resample(time="1D").max(), progress_bar=True
)
loaded_daily_data = add_wrf_crs(loaded_daily_data)
loaded_daily_data.to_netcdf(f"heat_index_{wl}.nc","w")
"""

'for wl in warming_levels[-1:]:\n## Retrieving data and sanity check plotting\nda1 = retrieve_clipped_data(\n    "NOAA Heat Index",\n    downscaling_method,\n    resolution,\n    "hourly",\n    "degF",\n    [wl],\n    gdf,\n    show_plots=True,\n    latitude=latitude,\n    longitude=longitude,\n)\nloaded_daily_data = load(\n    add_dummy_time_to_wl(da1).resample(time="1D").max(), progress_bar=True\n)\nloaded_daily_data = add_wrf_crs(loaded_daily_data)\nloaded_daily_data.to_netcdf(f"heat_index_{wl}.nc","w")\n'

## Load for all warming levels

In [27]:
import glob

filelist = glob.glob("heat_index_*.nc")
filelist.sort()
loaded_daily_data = xr.open_mfdataset(
    filelist, combine="nested", concat_dim="warming_level"
)
loaded_daily_data = add_wrf_crs(loaded_daily_data)

OSError: no files to open

In [ ]:
# Check that warming levels are in order
loaded_daily_data.warming_level.data

In [ ]:
# Apply land mask
loaded_daily_data["NOAA Heat Index"] = loaded_daily_data["NOAA Heat Index"].where(
    loaded_daily_data.landmask == 1, np.nan
)

In [ ]:
loaded_daily_data = load(loaded_daily_data)

In [ ]:
loaded_daily_data.simulation

# 1. 1-day event frequency

**Creating a metric for calculating the number of days above 90 for 1-day and 3-day consecutive heating day events**

In [ ]:
# Calculating metric for 1-day and 3-day events
event_durations = [1, 3]
threshold = 125
calc_datas = {}

In [ ]:
threshold_pairs = [[80, 90], [90, 103], [90, 200], [103, 124], [125, 200]]

#### Calculating and Visualizing Bins of 1-Day Events

In [ ]:
duration = 1
calc_thresh_pair_one_day = calc_thresh_pairs(
    loaded_daily_data["NOAA Heat Index"], threshold_pairs, duration
)

In [ ]:
plot_threshold_maps(
    calc_data=calc_thresh_pair_one_day,
    duration=duration,
    gdf=gdf_viz,
    longitude=longitude,
    latitude=latitude,
    lighter_r_rev=lighter_r_rev,
    hist_wl=hist_wl,
    lat_radius=lat_radius,
    lon_radius=lon_radius,
    lat_shift=lat_shift,
)

In [ ]:
tmp = loaded_daily_data.isel(time=0, warming_level=0, simulation=0).drop_vars(
    ["lakemask", "landmask", "lat", "lon"]
)
proj_mask = add_wrf_crs(xr.where(tmp.isnull(), 0, 1)).rio.reproject(dest_crs)

# Making a list of all the datasets to save
# in destination projection
ds_list = []
for tp in calc_thresh_pair_one_day.threshold_pair.data:
    # Calculating the metric data
    tmp = calc_thresh_pair_one_day.sel(threshold_pair=tp).drop_vars(
        ["lakemask", "landmask", "lat", "lon"]
    )
    tmp = add_wrf_crs(tmp).rio.reproject(dest_crs)
    ds_list.append(tmp)

# wrf_proj = tmp.rio.crs

# Export counts
gdf_list = []
fname_list = calc_thresh_pair_one_day.threshold_pair.data
for ds, fname in zip(ds_list, fname_list):
    ds = ds.where(proj_mask)
    ds.attrs["name"] = "NOAA Heat Index"
    # ds = add_wrf_crs(ds)
    gdf_new = export_reproj_data(ds, fname, dest_crs)
    if not gdf_new.empty:  # combine_exported_shapefiles will throw error if df empty
        gdf_list.append(fname + ".zip")

# Merge the geodataframes
wl_str = [str(wl) for wl in warming_levels]
merged = combine_exported_shapefiles(gdf_list, wl_str)

# Write merged geodataframe to file
export_shapefile_zip(
    merged, f"Exports/heat_index/1_{region_name}_1day_heating_events_count_annual"
)

# cleanup
for fname in gdf_list:
    os.remove(fname)

CRS.from_wkt('PROJCS["undefined",GEOGCS["undefined",DATUM["undefined",SPHEROID["undefined",6370000,0]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["standard_parallel_1",30],PARAMETER["standard_parallel_2",60],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",-70],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]')

# 2. 3-day event frequency

#### Calculating and Visualizing Bins of 3-Day Events

In [ ]:
duration = 3
calc_thresh_pair_three_day = calc_thresh_pairs(
    loaded_daily_data["NOAA Heat Index"], threshold_pairs, duration
)

In [ ]:
plot_threshold_maps(
    calc_data=calc_thresh_pair_three_day,
    duration=duration,
    gdf=gdf_viz,
    longitude=longitude,
    latitude=latitude,
    lighter_r_rev=lighter_r_rev,
    hist_wl=hist_wl,
    show_plots=False,
    lat_radius=lat_radius,
    lon_radius=lon_radius,
    lat_shift=lat_shift,
)

In [ ]:
tmp = loaded_daily_data.isel(time=0, warming_level=0, simulation=0).drop_vars(
    ["lakemask", "landmask", "lat", "lon"]
)
proj_mask = add_wrf_crs(xr.where(tmp.isnull(), 0, 1)).rio.reproject(dest_crs)

# Making a list of all the datasets to save
# in destination projection
ds_list = []
for tp in calc_thresh_pair_three_day.threshold_pair.data:
    # Calculating the metric data
    tmp = calc_thresh_pair_three_day.sel(threshold_pair=tp).drop_vars(
        ["lakemask", "landmask", "lat", "lon"]
    )
    tmp = tmp.rio.reproject(dest_crs)
    ds_list.append(tmp)

# wrf_proj = tmp.rio.crs

# Export counts
gdf_list = []
fname_list = calc_thresh_pair_three_day.threshold_pair.data
for ds, fname in zip(ds_list, fname_list):
    ds = ds.where(proj_mask)
    ds.attrs["name"] = "NOAA Heat Index"
    # ds = add_wrf_crs(ds)
    gdf_new = export_reproj_data(ds, fname, dest_crs)
    if not gdf_new.empty:  # combine_exported_shapefiles will throw error if df empty
        gdf_list.append(fname + ".zip")

# Merge the geodataframes
wl_str = [str(wl) for wl in warming_levels]
merged = combine_exported_shapefiles(gdf_list, wl_str)

# Write merged geodataframe to file
export_shapefile_zip(
    merged, f"Exports/heat_index/2_{region_name}_extreme_heat_3day_count_annual"
)

# cleanup
for fname in gdf_list:
    os.remove(fname)

# 3. 1-day event frequency by season

#### Calculating and Visualizing Bins of 1-Day Events by Season

In [ ]:
duration = 1
calc_thresh_pair_one_day_seasons = calc_thresh_pairs(
    loaded_daily_data["NOAA Heat Index"], threshold_pairs, duration, "season"
)

# Still by year value
nyears = len(np.unique(loaded_daily_data.time.dt.year))
calc_thresh_pair_one_day_seasons = calc_thresh_pair_one_day_seasons / nyears

In [ ]:
season_mapping = {"DJF": "winter", "MAM": "spring", "JJA": "summer", "SON": "fall"}

In [ ]:
for season in calc_thresh_pair_one_day_seasons.season.values:
    one_data = calc_thresh_pair_one_day_seasons.sel(season=season)
    plot_threshold_maps(
        calc_data=one_data,
        duration=duration,
        gdf=gdf_viz,
        longitude=longitude,
        latitude=latitude,
        lighter_r_rev=lighter_r_rev,
        hist_wl=hist_wl,
        season=season_mapping[season],
        show_plots=False,
        lat_radius=lat_radius,
        lon_radius=lon_radius,
        lat_shift=lat_shift,
    )

In [ ]:
tmp = loaded_daily_data.isel(time=0, warming_level=0, simulation=0).drop_vars(
    ["lakemask", "landmask", "lat", "lon"]
)
proj_mask = add_wrf_crs(xr.where(tmp.isnull(), 0, 1)).rio.reproject(dest_crs)

for season in calc_thresh_pair_one_day_seasons.season.values:
    # Making a list of all the datasets to save
    # in destination projection
    ds_list = []
    for tp in calc_thresh_pair_one_day_seasons.threshold_pair.data:
        # Calculating the metric data
        tmp = (
            calc_thresh_pair_one_day_seasons.sel(season=season)
            .sel(threshold_pair=tp)
            .drop_vars(["lakemask", "landmask", "lat", "lon"])
        )
        tmp = tmp.rio.reproject(dest_crs)
        ds_list.append(tmp)

    # wrf_proj = tmp.rio.crs

    # Export counts
    gdf_list = []
    fname_list = calc_thresh_pair_one_day_seasons.threshold_pair.data
    for ds, fname in zip(ds_list, fname_list):
        ds = ds.where(proj_mask)
        ds.attrs["name"] = "NOAA Heat Index"
        # ds = add_wrf_crs(ds)
        gdf_new = export_reproj_data(ds, fname, dest_crs)
        if (
            not gdf_new.empty
        ):  # combine_exported_shapefiles will throw error if df empty
            gdf_list.append(fname + ".zip")

    # Merge the geodataframes
    wl_str = [str(wl) for wl in warming_levels]
    merged = combine_exported_shapefiles(gdf_list, wl_str)

    # Write merged geodataframe to file
    export_shapefile_zip(
        merged, f"Exports/heat_index/3_{region_name}_1day_heating_events_count_{season}"
    )

    # cleanup
    for fname in gdf_list:
        os.remove(fname)

# 4. 3-day events by season

In [ ]:
duration = 3
calc_thresh_pair_three_day_seasons = calc_thresh_pairs(
    loaded_daily_data["NOAA Heat Index"], threshold_pairs, duration, "season"
)

# Still by year value
nyears = len(np.unique(loaded_daily_data.time.dt.year))
calc_thresh_pair_three_day_seasons = calc_thresh_pair_three_day_seasons / nyears

In [ ]:
season_mapping = {"DJF": "winter", "MAM": "spring", "JJA": "summer", "SON": "fall"}

In [ ]:
for season in calc_thresh_pair_three_day_seasons.season.values:
    one_data = calc_thresh_pair_three_day_seasons.sel(season=season)
    plot_threshold_maps(
        calc_data=one_data,
        duration=duration,
        gdf=gdf_viz,
        longitude=longitude,
        latitude=latitude,
        lighter_r_rev=lighter_r_rev,
        hist_wl=hist_wl,
        season=season_mapping[season],
        show_plots=False,
        lat_radius=lat_radius,
        lon_radius=lon_radius,
        lat_shift=lat_shift,
    )

In [ ]:
tmp = loaded_daily_data.isel(time=0, warming_level=0, simulation=0).drop_vars(
    ["lakemask", "landmask", "lat", "lon"]
)
proj_mask = add_wrf_crs(xr.where(tmp.isnull(), 0, 1)).rio.reproject(dest_crs)

for season in calc_thresh_pair_three_day_seasons.season.values:
    # Making a list of all the datasets to save
    # in destination projection
    ds_list = []
    for tp in calc_thresh_pair_three_day_seasons.threshold_pair.data:
        # Calculating the metric data
        tmp = (
            calc_thresh_pair_three_day_seasons.sel(season=season)
            .sel(threshold_pair=tp)
            .drop_vars(["lakemask", "landmask", "lat", "lon"])
        )
        tmp = tmp.rio.reproject(dest_crs)
        ds_list.append(tmp)

    # wrf_proj = tmp.rio.crs

    # Export counts
    gdf_list = []
    fname_list = calc_thresh_pair_three_day_seasons.threshold_pair.data
    for ds, fname in zip(ds_list, fname_list):
        ds = ds.where(proj_mask)
        ds.attrs["name"] = "NOAA Heat Index"
        # ds = add_wrf_crs(ds)
        gdf_new = export_reproj_data(ds, fname, dest_crs)
        if (
            not gdf_new.empty
        ):  # combine_exported_shapefiles will throw error if df empty
            gdf_list.append(fname + ".zip")

    # Merge the geodataframes
    wl_str = [str(wl) for wl in warming_levels]
    merged = combine_exported_shapefiles(gdf_list, wl_str)

    # Write merged geodataframe to file
    export_shapefile_zip(
        merged, f"Exports/heat_index/4_{region_name}_3day_heating_events_count_{season}"
    )

    # cleanup
    for fname in gdf_list:
        os.remove(fname)

#### Calculating and Visualizing days over 80F

In [ ]:
# Calculating metric for 1-day and 3-day events
threshold = 80
event_durations = [1, 3]

**Visualizing counts by season**

In [ ]:
event_duration = 1
# Visualizing counts by season
calc_data = calc_freq(
    data=loaded_daily_data["NOAA Heat Index"],
    threshold=threshold,
    duration=event_duration,
    comparison="greater",
    reduce_func="median",
    groupby="season",
)
# Still by year value
nyears = len(np.unique(loaded_daily_data.time.dt.year))
calc_data = calc_data / nyears

In [ ]:
for season in calc_data.season.values:
    season_name = seasons_dict[season]
    season_data = calc_data.sel(season=season)
    cbar_lim = get_cbar_lim(season_data)
    plot_data = plot_metric_by_wl(
        season_data,
        f"Avg Number of {event_duration}-Day Consecutive Heating \n Events Per Year Above {threshold}°F for {season_name.capitalize()}",
        gdf_viz,
        longitude,
        latitude,
        "Days Per Year",
        lighter_r_rev,
        save_name=f"figures/heat_index/5_{region_name}_extreme_heat_{threshold}deg_{event_duration}_day_count_{season_name}",
        nrows=2,
        ncols=2,
        colorbar_min=0,
        colorbar_max=season_data.max().data,
        basemap_style="map",
        lat_radius=lat_radius,
        lon_radius=lon_radius,
        lat_shift=lat_shift,
        edgecolor="black",
    )

    # Plotting the deltas between warming level calculations
    plot_data = plot_metric_deltas(
        season_data,
        f"Difference in Avg Number of {event_duration}-Day Consecutive Heating \n Events Per Year Above {threshold}°F between Future WL and WL {hist_wl} for {season_name.capitalize()}",
        hist_wl,
        gdf_viz,
        longitude,
        latitude,
        "Difference in Days Per Year",
        "RdBu_r",
        save_name=f"figures/heat_index/5_{region_name}_extreme_heat_{threshold}deg_{event_duration}_day_diff_{season_name}",
        nrows=1,
        ncols=3,
        colorbar_min=-cbar_lim,
        colorbar_max=cbar_lim,
        basemap_style="map",
        lat_radius=lat_radius,
        lon_radius=lon_radius,
        lat_shift=lat_shift,
        edgecolor="black",
    )

In [ ]:
tmp = loaded_daily_data.isel(time=0, warming_level=0, simulation=0).drop_vars(
    ["lakemask", "landmask", "lat", "lon"]
)
proj_mask = add_wrf_crs(xr.where(tmp.isnull(), 0, 1)).rio.reproject(dest_crs)

for season in calc_data.season.values:
    # Making a list of all the datasets to save
    # in destination projection
    ds_list = []
    tmp = calc_data.sel(season=season).drop_vars(["lakemask", "landmask", "lat", "lon"])
    tmp = tmp.rio.reproject(dest_crs)
    ds_list.append(tmp)

    # wrf_proj = tmp.rio.crs

    # Export counts
    gdf_list = []
    fname_list = [str(threshold)]
    for ds, fname in zip(ds_list, fname_list):
        ds = ds.where(proj_mask)
        ds.attrs["name"] = "NOAA Heat Index"
        # ds = add_wrf_crs(ds)
        gdf_new = export_reproj_data(ds, fname, dest_crs)
        if (
            not gdf_new.empty
        ):  # combine_exported_shapefiles will throw error if df empty
            gdf_list.append(fname + ".zip")

    # Merge the geodataframes
    wl_str = [str(wl) for wl in warming_levels]
    merged = combine_exported_shapefiles(gdf_list, wl_str)

    # Write merged geodataframe to file
    export_shapefile_zip(
        merged,
        f"Exports/heat_index/5_{region_name}_extreme_heat_{threshold}deg_{event_duration}day_count_{season}",
    )

    # cleanup
    for fname in gdf_list:
        os.remove(fname)